# DistilBERT : grille sur la validation (étape 4, cours 4)

**Avant tout** : menu *Exécution > Modifier le type d'exécution* > **GPU T4**, puis *Enregistrer*.

Le dossier Drive `MyDrive/review-intelligence-colab/` doit contenir `review_intelligence_colab.zip` (code + données).
Les résultats s'écrivent dans `MyDrive/review-intelligence-colab/resultats/` : ils survivent à une coupure de Colab.

Exécute les cellules dans l'ordre (Maj + Entrée). Ce soir : seulement la grille. **Pas de `final`** : il est verrouillé
tant que la grille n'a pas été expliquée et validée.

## 1. Le GPU est-il là ?
Tu dois voir une ligne « Tesla T4 ». Sinon : refaire *Modifier le type d'exécution*.

In [ ]:
!nvidia-smi

## 2. Brancher Google Drive
Une fenêtre demande l'autorisation d'accéder à ton Drive : accepte avec ton compte.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Déballer le paquet (code + données) sur le disque de la machine

In [ ]:
DOSSIER = "/content/drive/MyDrive/review-intelligence-colab"
!test -f "{DOSSIER}/review_intelligence_colab.zip" && echo "Paquet trouvé" || echo "PAQUET ABSENT : dépose review_intelligence_colab.zip dans {DOSSIER}"
!rm -rf /content/ri && mkdir -p /content/ri && unzip -q -o "{DOSSIER}/review_intelligence_colab.zip" -d /content/ri
%cd /content/ri
%env TOKENIZERS_PARALLELISM=false
!ls src data

## 4. Versions des bibliothèques (déjà installées sur Colab)

In [ ]:
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN")

## 5. Test de fumée (~1 à 2 min)
64 avis, 1 époque. Il vérifie que tout s'enchaîne : chargement, entraînement, mesure, écriture sur Drive.
Normal : un « LOAD REPORT » qui liste des poids UNEXPECTED (la tête de pré-entraînement, inutile ici) et MISSING
(`classifier`, `pre_classifier` : la tête à 5 sorties, neuve), et un avertissement sur HF_TOKEN.
Le score affiché (vers 0,1) ne veut rien dire : 64 avis et 1 époque servent seulement à vérifier la plomberie.

In [ ]:
!python -m src.distilbert grid --rapide --out-dir "{DOSSIER}/resultats"

## 6. La grille (reprise automatique)
Réglages (taux, époques du calendrier) : (2e-5, 4), (5e-5, 4), (5e-5, 6), chacun x 3 graines, mesure sur les
996 avis de validation à chaque époque. Les 6 entraînements du 14 oct. sont reconnus et sautés : seuls les
3 entraînements (5e-5, 6 époques) tournent, ~8 min. **Si Colab se coupe** : relance les cellules 2, 3 puis
celle-ci, la grille reprend où elle s'était arrêtée.

In [ ]:
!python -m src.distilbert grid --out-dir "{DOSSIER}/resultats"

## 7. Le choix sur la validation (à me coller demain, avec la sortie de la cellule 6)

In [ ]:
!cat "{DOSSIER}/resultats/choix.json"

## 8. Mesure finale sur le test (UNE seule fois, ~5 min)
Seulement après `GRILLE_VALIDEE = True` dans `src/distilbert.py`, décision commitée, **nouveau paquet** déposé
sur Drive et cellules 1 à 4 relancées. Ré-entraîne le réglage choisi sur les 4 979 avis, prédit les 300 avis de
test, écrit `resultats/preds_test_distilbert.jsonl` et enregistre le modèle dans `resultats/modele/`.
La commande refuse de tourner une deuxième fois.

In [ ]:
!python -m src.distilbert final --out-dir "{DOSSIER}/resultats"